# Wan 2.2 Img2Img — Фото из фото

**Генерация стилизованных изображений** с помощью Wan 2.2 14B + Lightning LoRA (4 шага!) + апскейл x2.

**Время настройки:** ~10-15 мин (скачивание моделей ~15 ГБ)
**VRAM:** ~14 ГБ пиковое потребление (T4 16 ГБ)

---

### Как это работает

```
[1. GPU]  →  [2. Установка]  →  [3. Ноды]  →  [4. Модели]  →  [5. Воркфлоу]  →  [6. ЗАГРУЗКА ФОТО]  →  [7. Запуск]
                                                                                          ↑
                                                                                 Самый важный шаг!
                                                                        Загрузите исходное фото для
                                                                        стилизации (любое фото/арт)
```

---

### Что делает этот ноутбук?

| Возможность | Описание |
|-------------|----------|
| **Img2Img** | Загрузите любое фото → получите стилизованную версию |
| **Lightning (4 шага)** | Генерация за 4 шага вместо 20-30, в 5-7 раз быстрее |
| **Апскейл x2** | Встроенный UltimateSDUpscale: 1280x1920 → 2560x3840 |
| **Гибкий denoise** | 0.3 = лёгкая стилизация, 0.7 = сильная переработка |

---

### Требования к входному фото

| Параметр | Рекомендация |
|----------|-------------|
| **Формат** | .jpg, .png |
| **Размер** | Любой (будет ресайзнуто до 1280x1920) |
| **Качество** | Чёткое, без водяных знаков |
| **Тип** | Портреты, пейзажи, арт — всё подходит |

### Чего НЕ нужно

- Видео или аудио файлы
- GIF-анимации
- Файлы больше 20 МБ

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

print("\nComfyUI установлен!")

In [ ]:
#@title 3. Установка кастомных нод
%cd /content/ComfyUI/custom_nodes

# Основные ноды
repos = {
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "ComfyUI-mxToolkit": "https://github.com/Smirnov75/ComfyUI-mxToolkit.git",
    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",
    "ComfyUI-Custom-Scripts": "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    "ComfyUI-wanBlockswap": "https://github.com/orssorbit/ComfyUI-wanBlockswap.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "ComfyUI_UltimateSDUpscale": "https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git",
    "ComfyUI-Image-Saver": "https://github.com/alexopus/ComfyUI-Image-Saver.git",
}

for name, url in repos.items():
    !test -d {name} || git clone {url}

# Зависимости
import os
for name in repos:
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        !pip install -r {req} -q -c /tmp/numpy_constraint.txt

print(f"\n{len(repos)} кастомных нод установлено!")

In [ ]:
#@title 4. Скачивание моделей (~15 ГБ, 5-10 мин)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models/WAN", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)
os.makedirs(f"{M}/upscale_models", exist_ok=True)
os.makedirs(f"{M}/loras/WAN", exist_ok=True)

# Comfy-Org модели (нативный формат)
COMFY22 = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"
COMFY21 = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"
KIJAI = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"

files = {
    # Wan 2.2 T2V Low Noise 14B fp8 (~9.5 ГБ)
    f"{M}/diffusion_models/WAN/wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors":
        f"{COMFY22}/diffusion_models/wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors",
    # Текстовый энкодер UMT5 XXL fp8 (~5 ГБ)
    f"{M}/text_encoders/umt5-xxl-encoder-fp8-e4m3fn-scaled.safetensors":
        f"{COMFY22}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    # VAE (~200 МБ)
    f"{M}/vae/wan_2.1_vae.safetensors":
        f"{COMFY21}/vae/wan_2.1_vae.safetensors",
    # Апскейлер 4x ClearReality (~64 МБ)
    f"{M}/upscale_models/4x-ClearRealityV1.pth":
        "https://huggingface.co/Kim2091/ClearRealityV1/resolve/main/4x-ClearRealityV1.pth",
    # Lightning LoRA — 4 шага вместо 20 (~614 МБ)
    f"{M}/loras/WAN/Wan2.2-Lightning_T2V-A14B-4steps-lora_LOW_fp16.safetensors":
        f"{KIJAI}/LoRAs/Wan22-Lightning/old/Wan2.2-Lightning_T2V-v1.1-A14B-4steps-lora_LOW_fp16.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

# Валидация скачивания
for dest in files:
    if os.path.exists(dest) and os.path.getsize(dest) < 1024:
        os.remove(dest)
        print(f"  ОШИБКА: {os.path.basename(dest)} не скачан — перезапустите ячейку")

print("\nВсе модели готовы!")
!du -sh {M}/diffusion_models/WAN/* {M}/text_encoders/* {M}/vae/* {M}/upscale_models/* {M}/loras/WAN/*

In [ ]:
#@title 5. Скачивание воркфлоу
import os

REPO_RAW = "https://raw.githubusercontent.com/russiankendricklamar/comfyui-colab-toolkit/main/workflows"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

!wget -q -O "{WF_DIR}/photo_wan_img2img.json" "{REPO_RAW}/photo_wan_img2img.json"
print("Скачано: photo_wan_img2img.json")

print(f"\nВоркфлоу сохранён в {WF_DIR}")

In [ ]:
#@title 6. Загрузка исходного фото
#@markdown ## Нажмите кнопку "Выбрать файлы" ниже
#@markdown
#@markdown Загрузите фото, которое хотите стилизовать.
#@markdown Файл сохранится в `/content/ComfyUI/input/`.
#@markdown
#@markdown ---
#@markdown
#@markdown ### Какие фото подходят?
#@markdown
#@markdown | Тип | Подходит? | Пример |
#@markdown |-----|-----------|--------|
#@markdown | Портрет | Да | Фото лица, поясной портрет |
#@markdown | Пейзаж | Да | Природа, городские виды |
#@markdown | Арт/иллюстрация | Да | Рисунки, цифровой арт |
#@markdown | Фото с текстом | Нет | Скриншоты, мемы |

from google.colab import files
from IPython.display import display, Image as IPImage
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

VALID_EXT = ('.jpg', '.jpeg', '.png', '.webp')

print("=" * 50)
print("  ЗАГРУЗКА ФОТО ДЛЯ IMG2IMG")
print("=" * 50)
print(f"  Папка: {INPUT_DIR}")
print(f"  Форматы: {', '.join(VALID_EXT)}")
print()

uploaded = files.upload()

saved = []
skipped = []

for name, data in uploaded.items():
    ext = os.path.splitext(name)[1].lower()
    if ext not in VALID_EXT:
        skipped.append(name)
        continue
    path = os.path.join(INPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    saved.append(name)

print(f"\n{'=' * 50}")
print("  РЕЗУЛЬТАТ")
print(f"{'=' * 50}")

if saved:
    print(f"\n  Загружено: {len(saved)} файлов")
    for f in saved:
        print(f"    {f}")

if skipped:
    print(f"\n  Пропущено (неподдерживаемый формат): {len(skipped)}")
    for f in skipped:
        print(f"    {f}")

# Превью
if saved:
    print(f"\nПревью:")
    for f in saved[:4]:
        path = os.path.join(INPUT_DIR, f)
        try:
            display(IPImage(filename=path, width=300))
        except Exception:
            pass
    print("\nОткройте ComfyUI, загрузите воркфлоу и выберите это фото в ноде LoadImage")

In [ ]:
#@title 7. Запуск ComfyUI
#@markdown ---
#@markdown ### Способ подключения
#@markdown **ngrok** — самый стабильный (нужен бесплатный токен).
#@markdown Cloudflare и localtunnel — без регистрации, но менее стабильны.
tunnel_method = "ngrok" #@param ["ngrok", "cloudflare", "localtunnel"]
ngrok_token = "" #@param {type:"string"}
#@markdown > Токен ngrok: [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

import subprocess, time, re, os, requests, shutil

# Борьба с фрагментацией CUDA памяти
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Логи ComfyUI в файл для диагностики
LOG_FILE = "/content/comfyui.log"

# Убиваем предыдущий экземпляр ComfyUI если есть
!kill -9 $(lsof -t -i:8188) 2>/dev/null || true

# Запуск ComfyUI с CORS для корректной работы через туннель
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
     '--enable-cors-header', '*'],
    cwd='/content/ComfyUI',
    stdout=open(LOG_FILE, 'w'),
    stderr=subprocess.STDOUT
)

print("Запуск ComfyUI...")
started = False
for i in range(24):
    time.sleep(5)
    try:
        r = requests.get("http://localhost:8188/api/system_stats", timeout=3)
        if r.status_code == 200:
            print(f"  ComfyUI запущен за {(i+1)*5} сек!")
            started = True
            break
    except Exception:
        pass

if not started:
    print("  ComfyUI не ответил за 120 сек.")
    print(f"\n  Логи ({LOG_FILE}):")
    with open(LOG_FILE) as f:
        lines = f.readlines()
        errors = [l for l in lines if any(w in l.lower() for w in ['error', 'traceback', 'exception', 'failed'])]
        if errors:
            print("  --- Ошибки ---")
            for l in errors[-15:]:
                print(f"  {l.rstrip()}")
        print("  --- Последние строки ---")
        for l in lines[-10:]:
            print(f"  {l.rstrip()}")
    raise RuntimeError("ComfyUI не запустился. Проверьте ошибки выше.")

# Проверка ошибок в логах даже при успешном запуске
with open(LOG_FILE) as f:
    log_text = f.read()
import_errors = [l for l in log_text.split('\n') if 'cannot import' in l.lower() or 'no module named' in l.lower()]
if import_errors:
    print(f"\n  Предупреждения при загрузке нод ({len(import_errors)}):")
    for l in import_errors[:5]:
        print(f"    {l.strip()}")

# === ТУННЕЛЬ ===
url = None

if tunnel_method == "ngrok":
    !pip install pyngrok -q
    from pyngrok import ngrok
    if not ngrok_token:
        import getpass
        ngrok_token = getpass.getpass("Введите ngrok authtoken (dashboard.ngrok.com → Your Authtoken): ")
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(8188)
    url = tunnel.public_url

elif tunnel_method == "cloudflare":
    if not shutil.which("cloudflared"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb 2>&1 | tail -1

    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    time.sleep(5)

    for _ in range(20):
        line = tunnel_proc.stdout.readline().decode()
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break

elif tunnel_method == "localtunnel":
    !npm install -g localtunnel > /dev/null 2>&1

    tunnel_proc = subprocess.Popen(
        ['lt', '--port', '8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    time.sleep(8)

    for _ in range(20):
        line = tunnel_proc.stdout.readline().decode()
        match = re.search(r'https://[\w-]+\.loca\.lt', line)
        if match:
            url = match.group(0)
            break

if url:
    print(f"\n{'='*60}")
    print(f"  ComfyUI готов!")
    print(f"  Откройте: {url}")
    print(f"{'='*60}")
    if tunnel_method == "ngrok":
        print(f"\n  ngrok покажет страницу 'Visit Site' при первом входе — нажмите кнопку.")
    elif tunnel_method == "localtunnel":
        print(f"\n  Localtunnel попросит пароль при первом входе.")
        print(f"  Пароль — ваш IP. Нажмите ссылку на странице.")
    print(f"\n  1. Меню -> Load -> photo_wan_img2img.json")
    print(f"  2. В ноде LoadImage выберите загруженное фото")
    print(f"  3. Напишите промпт и нажмите Queue Prompt")
    print(f"\n  Если страница белая:")
    print(f"    1. Обновите страницу (Ctrl+R)")
    print(f"    2. Подождите 10-15 секунд")
    print(f"    3. Если не помогло — попробуйте другой tunnel_method")
    print(f"\n  Логи ComfyUI: !cat {LOG_FILE}")
else:
    print(f"\nURL туннеля не найден.")
    print(f"  Попробуйте другой tunnel_method")
    print(f"  Логи: !cat {LOG_FILE}")

In [ ]:
#@title 8. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/img2img"
os.makedirs(dst, exist_ok=True)

results = glob.glob(f"{src}/**/*.png", recursive=True) + glob.glob(f"{src}/**/*.jpg", recursive=True)

if not results:
    print("Результатов пока нет. Сначала сгенерируйте изображение!")
else:
    for f in results:
        shutil.copy2(f, dst)
        print(f"Скопировано: {os.path.basename(f)}")
    print(f"\n{len(results)} файлов сохранено в {dst}")

---

## Гайд по воркфлоу `photo_wan_img2img`

### Основные параметры

| Параметр | Значение | Описание |
|----------|----------|----------|
| `denoise` | 0.3–0.9 | Сила изменений. 0.3 = лёгкая стилизация, 0.7 = средняя, 0.9 = почти полная перерисовка |
| `steps` | 4 | Lightning LoRA = 4 шага (не меняйте) |
| `CFG` | 1.0 | Для Lightning = 1.0 (не меняйте) |
| `NAG Scale` | 11 | Нормализованное внимание (можно оставить) |
| `resolution` | 1280x1920 | Портретная ориентация |
| `upscale_ratio` | 2x | 1280x1920 → 2560x3840 |
| `blocks_to_swap` | 40 | Для T4. Уменьшите для более быстрого GPU |

### Примеры промптов

```
anime style illustration, vibrant colors, detailed background, studio ghibli aesthetic
```

```
oil painting, impressionist style, thick brushstrokes, warm golden light
```

```
cyberpunk photography, neon lights, rain reflections, cinematic composition
```

### Если получается OOM (нехватка памяти)

1. Уменьшите разрешение в ноде ImageResize
2. Отключите группу Upscale через Fast Groups Bypasser
3. Увеличьте blocks_to_swap до 40 (максимум)